# Assignment 5 — Building a Robust ETL Pipeline with PostgreSQL & psycopg2

**Student:** Nawaraj Tamang

Extends the class14 banking ETL pipeline (`DBConnection`, `create_schema`,
`extract`, `transform`, `load_summary`) to add loan tracking, connection
hardening with retries, an updated categorization rule, a reconciliation
report, and unit tests for the pure transformation logic.

Runs against the real banking dataset (201 customers, 280 accounts, 4,613
transactions) rather than made-up sample rows — the same data used in the
Day 9 SQL practice notebook.


In [1]:
"""
Assignment 5 - Building a Robust ETL Pipeline with PostgreSQL & psycopg2
Nawaraj Tamang
"""

import logging
import os
import time

import psycopg2
from psycopg2 import OperationalError, Error as Psycopg2Error
from psycopg2.extras import RealDictCursor
from dotenv import load_dotenv

logging.basicConfig(
    filename="etl.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("banking.etl")
print("Logging configured -> etl.log")

Logging configured -> etl.log


## Credentials

Loaded from a git-ignored `.env` file, never hardcoded. A placeholder is shown below — see `.env.example` in the submission.

In [2]:
load_dotenv(override=True)  # looks for a ".env" file relative to this notebook

DB_HOST = os.getenv("DB_HOST")       # e.g. "127.0.0.1" (placeholder shown here)
DB_PORT = os.getenv("DB_PORT")       # e.g. "5432"
DB_NAME = os.getenv("DB_NAME")       # e.g. "banking_etl"
DB_USER = os.getenv("DB_USER")       # e.g. "postgres"
DB_PASSWORD = os.getenv("DB_PASSWORD")  # never printed or committed

_required = {"DB_HOST": DB_HOST, "DB_PORT": DB_PORT, "DB_NAME": DB_NAME,
             "DB_USER": DB_USER, "DB_PASSWORD": DB_PASSWORD}
_missing = [k for k, v in _required.items() if not v]
if _missing:
    raise RuntimeError(
        f"Missing required .env variable(s): {', '.join(_missing)}. "
        f"Please check your .env file."
    )
print("Credentials loaded from .env")

Credentials loaded from .env


## Task 2 — Hardened `DBConnection` and retry decorator

`DBConnection` now supports `reconnect()`. The `with_retry` decorator wraps
any function whose first argument is a `db` object: on `psycopg2.OperationalError`
it logs the attempt, always calls `db.rollback()` (or reconnects if the
connection itself is dead) before retrying, and gives up after 3 attempts.

In [3]:
class DBConnection:
    """Small wrapper so every part of the pipeline shares one connection."""

    def __init__(self, host, port, dbname, user, password):
        self.host, self.port, self.dbname, self.user, self.password = (
            host, port, dbname, user, password
        )
        self._connect()

    def _connect(self):
        self.conn = psycopg2.connect(
            host=self.host, port=self.port, dbname=self.dbname,
            user=self.user, password=self.password,
        )
        logger.info("Connected to database '%s'", self.dbname)

    def reconnect(self):
        """Used by the retry decorator when the connection itself has died."""
        try:
            self.conn.close()
        except Exception:
            pass
        self._connect()
        logger.info("Reconnected to database '%s'", self.dbname)

    def cursor(self):
        return self.conn.cursor(cursor_factory=RealDictCursor)

    def commit(self):
        self.conn.commit()

    def rollback(self):
        self.conn.rollback()

    def close(self):
        self.conn.close()
        logger.info("Connection closed")


def with_retry(max_attempts=3, delay_seconds=1):
    """Retries a db-operation function up to max_attempts times on
    psycopg2.OperationalError, always rolling back (or reconnecting)
    before each retry so the connection never sits in a broken state.
    The decorated function's first positional argument must be the db object.
    """
    def decorator(func):
        def wrapper(db, *args, **kwargs):
            attempt = 0
            while True:
                attempt += 1
                try:
                    return func(db, *args, **kwargs)
                except OperationalError as e:
                    logger.error(
                        "Attempt %d/%d for %s failed: %s",
                        attempt, max_attempts, func.__name__, e,
                    )
                    try:
                        db.rollback()
                    except Exception:
                        # Connection itself is dead (not just the transaction) - reconnect.
                        try:
                            db.reconnect()
                        except Exception as reconnect_err:
                            logger.error("Reconnect failed: %s", reconnect_err)
                    if attempt >= max_attempts:
                        logger.error(
                            "Giving up on %s after %d attempts", func.__name__, max_attempts
                        )
                        raise
                    time.sleep(delay_seconds)
        return wrapper
    return decorator


db = DBConnection(host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD)
print("Connected")

Connected


## Task 1 — Extended schema

`customers`, `accounts`, and `transactions` are the original class14 tables
(the same three-table banking schema used throughout the course, loaded
from real CSV data). This assignment adds:

- `loans` — one row per loan, linked to `customers` via a foreign key
- two new columns on `account_summary`: `total_loan_exposure`, `active_loan_count`

All DDL uses `CREATE TABLE IF NOT EXISTS`, matching the idempotent style
from `create_schema` in the original notebook. `accounts.customer_id` is a
plain inline foreign key to `customers` — the dataset's one intentionally
orphaned account row has been removed for now, so no `NOT VALID` workaround
is needed here.

In [4]:
DDL_TABLES = """
CREATE TABLE IF NOT EXISTS customers (
    customer_id     INT PRIMARY KEY,
    first_name      VARCHAR(50),
    last_name       VARCHAR(50),
    gender          VARCHAR(10),
    dob             DATE,
    city            VARCHAR(50),
    state           VARCHAR(50),
    country         VARCHAR(50),
    phone           VARCHAR(20),
    email           VARCHAR(100),
    occupation      VARCHAR(50),
    annual_income   NUMERIC(14, 2),
    credit_score    INT,
    kyc_status      VARCHAR(20),
    join_date       DATE,
    risk_category   VARCHAR(20)
);

CREATE TABLE IF NOT EXISTS accounts (
    account_id          INT PRIMARY KEY,
    customer_id         INT REFERENCES customers(customer_id),
    account_type        VARCHAR(30),
    branch              VARCHAR(50),
    ifsc_code           VARCHAR(15),
    currency             VARCHAR(5),
    balance              NUMERIC(14, 2),
    interest_rate        NUMERIC(5, 2),
    open_date            DATE,
    close_date           DATE,
    status               VARCHAR(20),
    is_joint_account     BOOLEAN
);

CREATE TABLE IF NOT EXISTS transactions (
    transaction_id   INT PRIMARY KEY,
    account_id       INT REFERENCES accounts(account_id),
    txn_date         DATE,
    txn_time         TIME,
    txn_type         VARCHAR(30),
    channel          VARCHAR(30),
    amount           NUMERIC(14, 2),
    currency         VARCHAR(5),
    balance_after    NUMERIC(14, 2),
    merchant         VARCHAR(60),
    description      VARCHAR(100),
    is_flagged       BOOLEAN
);

CREATE TABLE IF NOT EXISTS loans (
    loan_id         SERIAL PRIMARY KEY,
    customer_id     INTEGER REFERENCES customers(customer_id),
    principal       NUMERIC(12, 2) NOT NULL,
    interest_rate   NUMERIC(5, 2) NOT NULL,
    status          TEXT NOT NULL CHECK (status IN ('active', 'paid_off', 'defaulted')),
    start_date      DATE NOT NULL DEFAULT CURRENT_DATE
);

CREATE TABLE IF NOT EXISTS account_summary (
    customer_id             INT PRIMARY KEY REFERENCES customers(customer_id),
    total_transactions      NUMERIC(14, 2) NOT NULL DEFAULT 0,
    high_value_flag         BOOLEAN NOT NULL DEFAULT FALSE,
    category                TEXT NOT NULL DEFAULT 'Basic',
    total_loan_exposure     NUMERIC(14, 2) NOT NULL DEFAULT 0,
    active_loan_count       INTEGER NOT NULL DEFAULT 0
);
"""


@with_retry()
def create_schema(db):
    try:
        with db.cursor() as cur:
            cur.execute(DDL_TABLES)
        db.commit()
        logger.info("Schema created/verified (customers, accounts, transactions, loans, account_summary)")
        print("Schema created/verified")
    except Psycopg2Error as e:
        db.rollback()
        logger.error("Schema creation failed: %s", e)
        raise


create_schema(db)

Schema created/verified


## Load real data

Real CSV data (201 customers, 279 accounts, 4,613 transactions) is
bulk-loaded with `COPY ... WITH (FORMAT csv, HEADER true, NULL '')`,
exactly as in the original class notebook, rather than a handful of
made-up rows. The one intentionally orphaned account row has been removed
from the dataset for now.

For the new `loans` table, I seeded 6 loan rows across 3 real customers
from this dataset (`customer_id` 1, 2, 3), covering all three statuses
(`active`, `paid_off`, `defaulted`), per the assignment scenario.

In [5]:
def truncate_all(db):
    """Wipes existing rows so this notebook is safely re-runnable."""
    with db.cursor() as cur:
        cur.execute(
            "TRUNCATE TABLE account_summary, loans, transactions, accounts, customers RESTART IDENTITY CASCADE;"
        )
    db.commit()
    logger.info("Truncated all tables before reseeding")


def load_csv(db, table: str, csv_path: str):
    with db.cursor() as cur:
        with open(csv_path, "r", encoding="utf-8") as f:
            cur.copy_expert(
                f"COPY {table} FROM STDIN WITH (FORMAT csv, HEADER true, NULL '')",
                f,
            )
    db.commit()
    with db.cursor() as cur:
        cur.execute(f"SELECT COUNT(*) AS n FROM {table};")
        count = cur.fetchone()["n"]
    logger.info("Loaded %s rows into %s from %s", count, table, csv_path)
    print(f"Loaded {count} rows into {table}")


def seed_loans(db):
    """6 loan rows across 3 real customers (customer_id 1, 2, 3),
    covering all three statuses."""
    with db.cursor() as cur:
        cur.executemany(
            "INSERT INTO loans (customer_id, principal, interest_rate, status) VALUES (%s, %s, %s, %s);",
            [
                (1, 500000.00, 7.5, "active"),
                (1, 200000.00, 6.0, "paid_off"),
                (2, 1500000.00, 8.25, "active"),
                (2, 300000.00, 9.0, "active"),
                (3, 1000000.00, 10.5, "defaulted"),
                (3, 400000.00, 7.0, "active"),
            ],
        )
    db.commit()
    logger.info("Seeded 6 loan rows across 3 real customers (ids 1, 2, 3)")
    print("Seed data inserted: 6 loan rows across customer_id 1, 2, 3")


try:
    truncate_all(db)
    load_csv(db, "customers", "data/customers.csv")
    load_csv(db, "accounts", "data/accounts.csv")
    load_csv(db, "transactions", "data/transactions.csv")
    seed_loans(db)
    print("All data loaded successfully.")
    logger.info("Data load complete")
except Psycopg2Error as e:
    db.rollback()
    logger.error("Data load failed: %s", e)
    raise

Loaded 201 rows into customers
Loaded 279 rows into accounts
Loaded 4613 rows into transactions
Seed data inserted: 6 loan rows across customer_id 1, 2, 3
All data loaded successfully.


## Extract

Pulls raw rows out of the database as plain lists of dicts — no business
logic here, that all lives in `transform_logic.py` (Task 3) so it can be
unit tested without a database. Transactions only carry `account_id`, so
`extract_transactions` joins through `accounts` to attach each
transaction's `customer_id`.

In [6]:
@with_retry()
def extract_customer_ids(db):
    with db.cursor() as cur:
        cur.execute("SELECT customer_id FROM customers ORDER BY customer_id;")
        rows = cur.fetchall()
    logger.info("Extracted %d customer id(s)", len(rows))
    return [r["customer_id"] for r in rows]


@with_retry()
def extract_transactions(db):
    with db.cursor() as cur:
        cur.execute("""
            SELECT a.customer_id, t.amount, t.txn_date
            FROM transactions t
            JOIN accounts a ON t.account_id = a.account_id
            WHERE a.customer_id IS NOT NULL;
        """)
        rows = [dict(r) for r in cur.fetchall()]
    logger.info("Extracted %d transaction row(s)", len(rows))
    return rows


@with_retry()
def extract_loans(db):
    with db.cursor() as cur:
        cur.execute("SELECT customer_id, principal, interest_rate, status FROM loans;")
        rows = [dict(r) for r in cur.fetchall()]
    logger.info("Extracted %d loan row(s)", len(rows))
    return rows


customer_ids = extract_customer_ids(db)
transaction_rows = extract_transactions(db)
loan_rows = extract_loans(db)
print(f"customer_ids: {len(customer_ids)} customers")
print(f"transaction_rows: {len(transaction_rows)} rows")
print(f"loan_rows: {len(loan_rows)} rows")

customer_ids: 201 customers
transaction_rows: 4613 rows
loan_rows: 6 rows


## Task 3 — Transform

Pure business logic lives in `transform_logic.py`: `transform()`,
`transform_loans()`, `high_value_flag()`, and `categorize()` (now
loan-aware — any defaulted loan forces `"At Risk"` regardless of
transaction total). `build_account_summary()` combines them into one
row per customer, ready for `load_summary`.

In [7]:
from transform_logic import build_account_summary
from collections import Counter

summary_rows = build_account_summary(customer_ids, transaction_rows, loan_rows)
print(f"Built {len(summary_rows)} account_summary row(s)\n")

print("Category distribution:", dict(Counter(r["category"] for r in summary_rows)))
print()

print("Preview (first 5 rows):")
for row in summary_rows[:5]:
    print(row)

print()
print("The 3 customers with seeded loans (ids 1, 2, 3):")
for row in summary_rows:
    if row["customer_id"] in (1, 2, 3):
        print(row)

Built 201 account_summary row(s)

Category distribution: {'Premium': 198, 'At Risk': 1, 'Standard': 1, 'Basic': 1}

Preview (first 5 rows):
{'customer_id': 1, 'total_transactions': 748232.0, 'total_loan_exposure': 500000.0, 'active_loan_count': 1, 'high_value_flag': True, 'category': 'Premium'}
{'customer_id': 2, 'total_transactions': 733676.4, 'total_loan_exposure': 1800000.0, 'active_loan_count': 2, 'high_value_flag': True, 'category': 'Premium'}
{'customer_id': 3, 'total_transactions': 764547.4099999999, 'total_loan_exposure': 400000.0, 'active_loan_count': 1, 'high_value_flag': True, 'category': 'At Risk'}
{'customer_id': 4, 'total_transactions': 994561.3799999999, 'total_loan_exposure': 0.0, 'active_loan_count': 0, 'high_value_flag': True, 'category': 'Premium'}
{'customer_id': 5, 'total_transactions': 771171.1799999999, 'total_loan_exposure': 0.0, 'active_loan_count': 0, 'high_value_flag': True, 'category': 'Premium'}

The 3 customers with seeded loans (ids 1, 2, 3):
{'customer_i

## Task 4 — Load (upsert) and verify

In [8]:
@with_retry()
def load_summary(db, rows):
    upsert_sql = """
        INSERT INTO account_summary
            (customer_id, total_transactions, high_value_flag, category,
             total_loan_exposure, active_loan_count)
        VALUES (%(customer_id)s, %(total_transactions)s, %(high_value_flag)s,
                %(category)s, %(total_loan_exposure)s, %(active_loan_count)s)
        ON CONFLICT (customer_id) DO UPDATE SET
            total_transactions  = EXCLUDED.total_transactions,
            high_value_flag     = EXCLUDED.high_value_flag,
            category            = EXCLUDED.category,
            total_loan_exposure = EXCLUDED.total_loan_exposure,
            active_loan_count   = EXCLUDED.active_loan_count;
    """
    try:
        with db.cursor() as cur:
            for row in rows:
                cur.execute(upsert_sql, row)
        db.commit()
        logger.info("Upserted %d row(s) into account_summary", len(rows))
        print(f"Upserted {len(rows)} row(s) into account_summary")
    except Psycopg2Error as e:
        db.rollback()
        logger.error("load_summary failed: %s", e)
        raise


load_summary(db, summary_rows)

Upserted 201 row(s) into account_summary


In [9]:
@with_retry()
def verify(db):
    """Reconciliation report: per customer, total transactions, total loan
    exposure, active loan count, and final category. Confirms every
    customer appears exactly once in account_summary."""
    with db.cursor() as cur:
        cur.execute("SELECT COUNT(*) AS n FROM customers;")
        n_customers = cur.fetchone()["n"]

        cur.execute("SELECT COUNT(*) AS n FROM account_summary;")
        n_summary = cur.fetchone()["n"]

        cur.execute("""
            SELECT c.customer_id, c.first_name || ' ' || c.last_name AS name,
                   s.total_transactions, s.total_loan_exposure,
                   s.active_loan_count, s.category
            FROM customers c
            JOIN account_summary s ON s.customer_id = c.customer_id
            ORDER BY c.customer_id;
        """)
        report_rows = cur.fetchall()

    print("=" * 84)
    print(f"RECONCILIATION REPORT  (showing first 10 of {len(report_rows)} customers)")
    print("=" * 84)
    print(f"{'ID':<4}{'Name':<20}{'Total Txns':>14}{'Loan Exposure':>16}{'Active Loans':>14}{'Category':>12}")
    for r in report_rows[:10]:
        print(f"{r['customer_id']:<4}{r['name']:<20}{float(r['total_transactions']):>14.2f}"
              f"{float(r['total_loan_exposure']):>16.2f}{r['active_loan_count']:>14}{r['category']:>12}")
    print("...")
    print("\nCustomers with seeded loans (ids 1, 2, 3):")
    for r in report_rows:
        if r["customer_id"] in (1, 2, 3):
            print(f"{r['customer_id']:<4}{r['name']:<20}{float(r['total_transactions']):>14.2f}"
                  f"{float(r['total_loan_exposure']):>16.2f}{r['active_loan_count']:>14}{r['category']:>12}")
    print("-" * 84)
    match = n_customers == n_summary == len(report_rows)
    status = "OK - every customer appears exactly once" if match else "MISMATCH"
    print(f"customers: {n_customers}  |  account_summary rows: {n_summary}  |  {status}")
    logger.info(
        "Verification: %d customers, %d account_summary rows, match=%s",
        n_customers, n_summary, match,
    )
    return report_rows


_ = verify(db)

RECONCILIATION REPORT  (showing first 10 of 201 customers)
ID  Name                    Total Txns   Loan Exposure  Active Loans    Category
1   Krishna Rai              748232.00       500000.00             1     Premium
2   Anita Lama               733676.40      1800000.00             2     Premium
3   Gita Pandey              764547.41       400000.00             1     At Risk
4   Mina Pandey              994561.38            0.00             0     Premium
5   Indira Gurung            771171.18            0.00             0     Premium
6   Sabina Bhattarai        1401483.37            0.00             0     Premium
7   Suresh Tamang              7985.85            0.00             0    Standard
8   Amit Chaudhary           404995.42            0.00             0     Premium
9   Arjun Bhandari          1017961.36            0.00             0     Premium
10  Sita Maharjan            910532.37            0.00             0     Premium
...

Customers with seeded loans (ids 1, 2, 3):
1 

## Task 2 — Demonstrating the retry logic

To prove the retry/reconnect logic actually works (not just that it's
written), this cell forces a **genuine** `psycopg2.OperationalError` by
having a second admin connection terminate our pipeline connection's
backend process mid-pipeline (`pg_terminate_backend`), then immediately
calls a `@with_retry`-decorated extract function. The first attempt fails
with a real `OperationalError`, the decorator reconnects, and the retry
succeeds. Check `etl.log` for the attempt-by-attempt trail.

In [10]:
admin_conn = psycopg2.connect(host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD)
admin_conn.autocommit = True

backend_pid = db.conn.get_backend_pid()
with admin_conn.cursor() as admin_cur:
    admin_cur.execute("SELECT pg_terminate_backend(%s);", (backend_pid,))
admin_conn.close()

print(f"Simulated failure: terminated backend pid {backend_pid} mid-pipeline.")
print("Calling extract_transactions(db) - expect a logged retry, then success...")

result = extract_transactions(db)
print(f"Recovered: extract_transactions returned {len(result)} row(s) after retry.")

Simulated failure: terminated backend pid 330 mid-pipeline.
Calling extract_transactions(db) - expect a logged retry, then success...


Recovered: extract_transactions returned 4613 row(s) after retry.


In [11]:
with open("etl.log") as f:
    lines = f.readlines()
print("".join(lines[-12:]))

2026-09-13 12:05:07,176 | INFO | Seeded 6 loan rows across 3 real customers (ids 1, 2, 3)
2026-09-13 12:05:07,177 | INFO | Data load complete
2026-09-13 12:05:07,188 | INFO | Extracted 201 customer id(s)
2026-09-13 12:05:07,250 | INFO | Extracted 4613 transaction row(s)
2026-09-13 12:05:07,250 | INFO | Extracted 6 loan row(s)
2026-09-13 12:05:07,281 | INFO | Upserted 201 row(s) into account_summary
2026-09-13 12:05:07,292 | INFO | Verification: 201 customers, 201 account_summary rows, match=True
2026-09-13 12:05:07,313 | ERROR | Attempt 1/3 for extract_transactions failed: SSL connection has been closed unexpectedly

2026-09-13 12:05:07,326 | INFO | Connected to database 'banking_etl'
2026-09-13 12:05:07,326 | INFO | Reconnected to database 'banking_etl'
2026-09-13 12:05:08,344 | INFO | Extracted 4613 transaction row(s)



## Close the connection

In [12]:
db.close()
print("Connection closed. See etl.log for the full operation history, including the simulated retry.")

Connection closed. See etl.log for the full operation history, including the simulated retry.
